# Qwen2-VL reminders: no hint vs object list vs ranked hint (Kaggle, GPU)
For every clip, Qwen2-VL-2B writes one walking reminder under three conditions; only the extra text differs:
- `none`: the model alone (general instruction).
- `list`: the 8 nearest detected objects, in alphabetical order (no ranking, no decision).
- `ranked`: our module's hint, written as a sentence.

`MODE = "pilot"` runs 10 dev_00 clips per decision level (used to choose the prompt); `"full"` runs every clip.
Outputs are saved after every clip, so a rerun continues where it stopped.

## 1. Settings

In [ ]:
import json, re, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from IPython.display import FileLink

SUBSET = "test"
MODE = "full" # "pilot" = 10 clips , "full" = every clip
PROMPT_VERSION = "v13" # final prompt, chosen on the dev_00 pilot
SEED = 42

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
FRAMES_USED = [0, 4, 8] # three frames ~1.3 s apart; frame 8 = now (the frame our module decides on)
MAX_PIXELS = 448 * 448 # caps each image at ~256 image tokens
MAX_NEW_TOKENS = 128
CONDITIONS = ["none", "list", "ranked"]
LIST_MAX = 8  # the 8 nearest detected objects, in alphabetical order
PRECISION = "auto"  # "auto" = float16 on T4/P100, bfloat16 on newer GPUs; "float32" if error

INPUT = Path("/kaggle/input")
OUT_FILE = Path(f"/kaggle/working/qwen/{SUBSET}/qwen_{MODE}_{PROMPT_VERSION}.jsonl")
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

## 2. The module's outputs and the frames
- Attach `wad-module-outputs` (one folder per subset) and the WAD frames dataset for the subset.

In [ ]:
# the hints: the hints.jsonl in the folder named after the subset
found = [p.parent for p in INPUT.rglob("hints.jsonl") if p.parent.name == SUBSET]
assert len(found) == 1, f"expected one hints.jsonl for {SUBSET}, found {len(found)}"
hints = pd.read_json(found[0] / "hints.jsonl", lines=True).set_index("clip_id")
ranking = pd.read_parquet(found[0] / "ranking.parquet")

# the frames: the 'frames' folder that contains our clips
FRAMES_DIR = next(p for p in INPUT.rglob("frames") if (p / hints.index[0]).is_dir())
frame_file = lambda clip, k: FRAMES_DIR / clip / f"{k}.jpg"
assert all(frame_file(c, k).exists() for c in hints.index for k in FRAMES_USED)
print(f"{len(hints)} clips | our levels: {hints.level.value_counts().to_dict()}")

## 3. Clips to run

In [ ]:
LEVELS = ["stop", "move", "slow", "caution", "clear"]
CLIPS = ([c for lvl in LEVELS for c in hints[hints.level == lvl].sample(min(10, (hints.level == lvl).sum()), random_state=SEED).index]
         if MODE == "pilot" else hints.index.tolist())
print(f"{MODE}: {len(CLIPS)} clips x {len(CONDITIONS)} conditions")

In [ ]:
# none: the model alone
BASE = (
    "You are helping a blind person walk safely. The images are {n} frames from a camera on their chest, "
    "about 1.3 seconds apart; the last image is now.\n"
    "Write one short walking reminder (one or two sentences). Name the most important hazard, its direction as "
    "a clock position (12 o'clock is straight ahead, 3 o'clock is right, 9 o'clock is left), its distance in "
    "steps if you can judge it, and what to do: keep going, slow down, move towards a clock direction, or stop. "
    "If nothing is in the way, say that the path ahead is clear."
)

# list and ranked: same instruction, different information
HINT_START = ("You are guiding a blind person who is walking. The images show what their chest camera sees; "
              "the last image is now.\nWhat is ahead of them{order}: {info}\n")
HINT_TASK = ("Now tell them in one or two short sentences: name the hazard, its clock direction and how many "
             "steps away, then say what to do. Use the images as well.")
HINT_TASK_CLEAR = "Now tell them in one short sentence."

MOTION = {"same direction": "walking the same way as you", "approaching": "coming towards you",
          "crossing": "crossing your path", "moving away": "moving away"}
PLURAL = {"stairs", "steps", "bollards", "people"}

def spoken(name):
    first = name.split()[0]
    return name if first in PLURAL else ("an " if first[0] in "aeiou" else "a ") + name

def hint_as_reminder(h):
    """Same content as our hint, written as a spoken reminder instead of a table."""
    if not re.search(r"^\d+\)", h, re.M): # the 'clear' hint
        m = re.search(r"about (\d+) steps", h)
        return f"The path ahead is clear for about {m.group(1)} steps. Keep going straight." if m else h
    items, action, ctx = [], "", ""
    for ln in h.split("\n"):
        m = re.match(r"\d+\)\s*(.+)", ln)
        if m: # "1) person | 12 o'clock | 7 steps | in your path | crossing"
            p = [x.strip() for x in m.group(1).split("|")]
            steps = re.sub(r"^1 steps$", "1 step", p[2])
            s = f"{spoken(p[0])} at {p[1]}, {steps} away, {p[3]}"
            if len(p) > 4:
                s += ", " + MOTION.get(p[4], p[4])
            items.append(s)
        elif ln.startswith("Action:"):
            action = ln[7:].strip().rstrip(".")
        elif ln.startswith("Context:"):
            ctx = ln[8:].strip()
    out = f"Watch out: {items[0]}."
    if action:
        out += f" {action[0].upper() + action[1:]}."
    if len(items) > 1:
        out += " Also " + "; ".join(items[1:]) + "."
    if "on the road" in ctx:
        out += " You are on the road."
    if "crowded" in ctx:
        out += " It is crowded."
    return out

def list_as_text(clip):
    """The nearest detected objects, alphabetical (no ranking)."""
    r = ranking[ranking.clip_id == clip].nsmallest(LIST_MAX, "dist_m")
    if len(r) == 0:
        return "nothing was detected."
    items = sorted(f"{spoken(p)} at {int(c)} o'clock, about {max(1, round(s))} steps" for p, c, s in zip(r.prompt, r.clock, r.steps))
    return "; ".join(items) + "."

def prompt_for(clip, condition):
    if condition == "none":
        return BASE.format(n=len(FRAMES_USED))
    if condition == "list":
        return HINT_START.format(order=" (in no particular order)", info=list_as_text(clip)) + HINT_TASK
    is_clear = hints.loc[clip, "level"] == "clear"
    return (HINT_START.format(order="", info=hint_as_reminder(hints.loc[clip, "hint"]))
            + (HINT_TASK_CLEAR if is_clear else HINT_TASK))

for c in CONDITIONS: # the three prompts for one clip
    print(f"===== {c} =====\n{prompt_for(CLIPS[0], c)}\n")

## 5. Load Qwen2-VL-2B

In [ ]:
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
DTYPE = ((torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)
         if PRECISION == "auto" else getattr(torch, PRECISION))
model = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="cuda").eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=128 * 28 * 28, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = "left" # needed to generate the three prompts in one batch

## 6. Generate (resumes if interrupted)
- The three conditions of a clip run together in one batch, with greedy decoding, so each sees the same images and settings.

In [ ]:
def generate(clip):
    imgs = [Image.open(frame_file(clip, k)).convert("RGB") for k in FRAMES_USED]
    texts = []
    for c in CONDITIONS:
        msg = [{"role": "user", "content": [{"type": "image"} for _ in imgs] + [{"type": "text", "text": prompt_for(clip, c)}]}]
        texts.append(processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True))
    inputs = processor(text=texts, images=imgs * len(CONDITIONS), padding=True, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    replies = processor.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return [r.strip() for r in replies]

done = {json.loads(l)["clip_id"] for l in OUT_FILE.open()} if OUT_FILE.exists() else set()
todo = [c for c in CLIPS if c not in done]
print(f"{len(done)} clips done, {len(todo)} to run")

t0 = time.time()
with OUT_FILE.open("a") as f:
    for i, clip in enumerate(todo, 1):
        t = time.time()
        replies = generate(clip)
        f.write(json.dumps({"clip_id": clip, "seconds": round(time.time() - t, 2),
                            **{f"qwen_{c}": r for c, r in zip(CONDITIONS, replies)}}) + "\n")
        f.flush()
        if i % 50 == 0 or i == len(todo):
            per = (time.time() - t0) / i
            print(f"  {i}/{len(todo)} clips | {per:.1f} s per clip | ~{per * (len(todo) - i) / 60:.0f} min left")

## 7. Quick look
- A form check only; scoring against the human reminders is done in the evaluation notebook.
- For `ranked`, saying "clear" when the module found a hazard should be rare (≤ 5% was the pilot rule).

In [ ]:
res = pd.read_json(OUT_FILE, lines=True).set_index("clip_id")
assert res.index.is_unique and set(CLIPS) <= set(res.index)
lv = hints.loc[res.index, "level"]

def form(t):
    t = t.astype(str).str.strip()
    return {"words (median)": t.str.split().str.len().median(),
            "cut off": (~t.str[-1:].isin([".", "!", "?"])).mean(),
            "says 'clear' when we found a hazard": t[lv != "clear"].str.contains("clear", case=False).mean(),
            "mentions the module/notes": t.str.contains(r"\b(?:module|notes?|detector|information)\b", case=False).mean(),
            "starts 'As you'": t.str.match(r"(?i)as (you|we)\b").mean(),
            "empty": (t == "").mean()}

pd.DataFrame({c: form(res[f"qwen_{c}"]) for c in CONDITIONS}).round(2)

In [ ]:
# three examples: our hint and the three replies
for clip in res.sample(3, random_state=SEED).index:
    print(f"=== {clip} (our level: {hints.loc[clip, 'level']})")
    print("  hint  :", hints.loc[clip, "hint"].replace("\n", " / "))
    for c in CONDITIONS:
        print(f"  {c:<6}:", res.loc[clip, f"qwen_{c}"])
    print()

## 8. Download
- Contains Qwen's replies only (no WAD text). Put it in `results/<subset>/qwen/` in the repo.

In [ ]:
FileLink(str(OUT_FILE.relative_to("/kaggle/working")))